# 05 — Extract emotion2vec utterance-level features

Revised version for IJAE revision.

Main changes:
- uses the new `newcode` project folder;
- uses `iic/emotion2vec_plus_base` consistently;
- saves checkpoint SHA256 and environment manifest;
- validates that extracted embeddings are 768-dimensional;
- saves crop/padding duration statistics;
- saves SHA256 manifest for generated feature files.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Clean compatible PyTorch stack for Colab
# Run this once after a fresh runtime, then restart runtime if Colab asks.

!pip -q uninstall -y torch torchvision torchaudio cuda-python cuda-bindings cuda-toolkit nvidia-cublas-cu13 nvidia-cuda-runtime-cu13 nvidia-cudnn-cu13

!pip -q install torch==2.11.0 torchvision==0.26.0 torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu128

!pip -q install "jedi>=0.16"
!pip -q install "pandas==2.2.2" "numpy<2.1" "scikit-learn" "librosa==0.11.0" "soundfile" "tqdm"
!pip -q install "funasr==1.4.1" "modelscope==1.39.1" --upgrade-strategy only-if-needed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.3/820.3 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 142.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 148.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, which is not installed.
pylibraft-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, which is not installed.
cuml-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, which is not installed.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1

In [3]:
import os
import json
import random
import warnings
import hashlib
import platform
import datetime
import importlib.metadata as im

import numpy as np
import pandas as pd
import librosa
import soundfile as sf

from pathlib import Path
from tqdm import tqdm
from collections import Counter

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# IMPORTANT:
# Use the original project folder requested by the user.
# ============================================================
BASE_PROJECT = Path("/content/drive/MyDrive/New Jurnal Cross")

CSV_DIR = BASE_PROJECT / "processed_intra_csv"

# Use a folder name that matches the actual checkpoint.
# If you use iic/emotion2vec_plus_base, keep this as e2v_plus_base.
OUT_DIR = BASE_PROJECT / "processed_intra_features_e2v_plus_base"
TMP_AUDIO_DIR = BASE_PROJECT / "tmp_e2v_16k_audio_plus_base"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["emodb", "ravdess", "resd"]

CSV_FILES = {
    "emodb": CSV_DIR / "split_emodb_6class_optimized.csv",
    "ravdess": CSV_DIR / "split_ravdess_6class_optimized.csv",
    "resd": CSV_DIR / "split_resd_6class_optimized.csv",
}

LABELS = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
]

LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

CONFIG = {
    "sample_rate": 16000,
    "duration": 4.0,
    "model_name": "iic/emotion2vec_plus_base",
    "granularity": "utterance",
    "extract_embedding": True,
    "expected_embedding_dim": 768,
    "labels": LABELS,
    "preprocessing": {
        "mono": True,
        "resample_hz": 16000,
        "fixed_duration_sec": 4.0,
        "longer_utterances": "center-cropped",
        "shorter_utterances": "reflect-padded if length > 1 sample, otherwise zero-padded",
        "augmentation": "none",
        "scaler": "none at feature extraction stage",
    },
}

print("BASE_PROJECT :", BASE_PROJECT)
print("CSV_DIR      :", CSV_DIR)
print("OUT_DIR      :", OUT_DIR)
print("TMP_AUDIO_DIR:", TMP_AUDIO_DIR)
print("Model        :", CONFIG["model_name"])

missing = []
for ds, path in CSV_FILES.items():
    print(f"{ds:8s}", path.exists(), path)
    if not path.exists():
        missing.append(str(path))

if missing:
    raise FileNotFoundError("Missing CSV files:\n" + "\n".join(missing))


BASE_PROJECT : /content/drive/MyDrive/New Jurnal Cross
CSV_DIR      : /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv
OUT_DIR      : /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_e2v_plus_base
TMP_AUDIO_DIR: /content/drive/MyDrive/New Jurnal Cross/tmp_e2v_16k_audio_plus_base
Model        : iic/emotion2vec_plus_base
emodb    True /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_emodb_6class_optimized.csv
ravdess  True /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_ravdess_6class_optimized.csv
resd     True /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_resd_6class_optimized.csv


In [4]:
from funasr import AutoModel

e2v_model = AutoModel(
    model=CONFIG["model_name"],
    trust_remote_code=True,
    device="cuda:0",      # change to "cpu" if GPU is unavailable
    disable_update=True   # important for reproducibility
)

print("Loaded model:", CONFIG["model_name"])

funasr version: 1.4.1.


2026-08-08 09:45:16,150 | INFO    | modelscope_hub.download | Downloading 11 files from iic/emotion2vec_plus_base@master


Downloading:   0%|          | 0/11 [00:00<?, ?file/s]

.gitattributes:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

.DS_Store:   0%|          | 0.00/6.15k [00:00<?, ?B/s]

...

model.pt:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.15k [00:00<?, ?B/s]

test.wav:   0%|          | 0.00/131k [00:00<?, ?B/s]

tokens.txt:   0%|          | 0.00/120 [00:00<?, ?B/s]

Loading remote code failed: model, No module named 'model'
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.weight, /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.bias, /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.weight, /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.bias, /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.2.0.weight, /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.2.0.bias, /root/.cache/modelscope/models/iic--emotion2vec_plus_base/s

In [5]:
def pkg_ver(package_name):
    try:
        return im.version(package_name)
    except Exception:
        return "not installed"


def sha256_of_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def find_modelscope_checkpoint(model_id):
    """
    Cari checkpoint ModelScope/FunASR secara robust.
    Untuk iic/emotion2vec_plus_base, biasanya file checkpoint adalah model.pt.
    """
    org, name = model_id.split("/", 1)

    candidate_roots = [
        Path.home() / ".cache" / "modelscope" / "hub" / "models" / org / name,
        Path("/root/.cache/modelscope/hub/models") / org / name,
        Path.home() / ".cache" / "modelscope" / "hub" / org / name,
        Path("/root/.cache/modelscope/hub") / org / name,
        Path.home() / ".cache" / "modelscope" / "models" / f"{org}--{name}",
        Path("/root/.cache/modelscope/models") / f"{org}--{name}",
    ]

    exts = [".pt", ".pth", ".bin", ".safetensors"]
    candidates = []

    for root in candidate_roots:
        if root.exists():
            for ext in exts:
                candidates.extend(root.rglob(f"*{ext}"))

    candidates = sorted(
        list({p.resolve() for p in candidates if p.is_file()}),
        key=lambda p: p.stat().st_size,
        reverse=True
    )

    if len(candidates) == 0:
        return None, []

    return candidates[0], candidates


checkpoint_path, checkpoint_candidates = find_modelscope_checkpoint(CONFIG["model_name"])

checkpoint_info = []
for p in checkpoint_candidates:
    checkpoint_info.append({
        "path": str(p),
        "name": p.name,
        "size_bytes": p.stat().st_size,
        "sha256": sha256_of_file(p),
    })

primary_checkpoint = checkpoint_info[0] if checkpoint_info else None

environment_manifest = {
    "notebook": "05_extract_emotion2vec_intra.ipynb",
    "task": "emotion2vec utterance-level feature extraction",
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "base_project": str(BASE_PROJECT),
    "csv_dir": str(CSV_DIR),
    "out_dir": str(OUT_DIR),
    "tmp_audio_dir": str(TMP_AUDIO_DIR),

    "model_id": CONFIG["model_name"],
    "granularity": CONFIG["granularity"],
    "extract_embedding": CONFIG["extract_embedding"],
    "expected_embedding_dim": CONFIG["expected_embedding_dim"],

    "checkpoint_path": str(checkpoint_path) if checkpoint_path is not None else None,
    "checkpoint_size_mb": round(checkpoint_path.stat().st_size / (1024 ** 2), 2) if checkpoint_path is not None else None,
    "checkpoint_sha256": sha256_of_file(checkpoint_path) if checkpoint_path is not None else None,
    "all_checkpoint_candidates": checkpoint_info,

    "config": CONFIG,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "funasr_version": pkg_ver("funasr"),
    "modelscope_version": pkg_ver("modelscope"),
    "torch_version": pkg_ver("torch"),
    "torchaudio_version": pkg_ver("torchaudio"),
    "librosa_version": pkg_ver("librosa"),
    "soundfile_version": pkg_ver("soundfile"),
    "numpy_version": pkg_ver("numpy"),
    "pandas_version": pkg_ver("pandas"),
    "scikit_learn_version": pkg_ver("scikit-learn"),
}

with open(OUT_DIR / "environment_manifest.json", "w") as f:
    json.dump(environment_manifest, f, indent=2)

print("=" * 80)
print("EMOTION2VEC REPRODUCIBILITY MANIFEST")
print("=" * 80)
print("Model ID          :", environment_manifest["model_id"])
print("Checkpoint path   :", environment_manifest["checkpoint_path"])
print("Checkpoint SHA256 :", environment_manifest["checkpoint_sha256"])
print("Checkpoint size   :", environment_manifest["checkpoint_size_mb"], "MB")
print("FunASR version    :", environment_manifest["funasr_version"])
print("ModelScope ver.   :", environment_manifest["modelscope_version"])
print("Torch version     :", environment_manifest["torch_version"])
print("Saved manifest    :", OUT_DIR / "environment_manifest.json")

if primary_checkpoint is None:
    print("\nWARNING: No checkpoint file was found in the expected ModelScope cache.")

EMOTION2VEC REPRODUCIBILITY MANIFEST
Model ID          : iic/emotion2vec_plus_base
Checkpoint path   : /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Checkpoint SHA256 : 60710b5aae1dbe69bdac8920028fb05882d4314fd09031922b4b61ee9e7aadbd
Checkpoint size   : 1066.44 MB
FunASR version    : 1.4.1
ModelScope ver.   : 1.39.1
Torch version     : 2.11.0+cu128
Saved manifest    : /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_e2v_plus_base/environment_manifest.json


In [6]:
def load_audio_fixed(path, sr=16000, duration=4.0, return_info=False):
    """
    Load audio as mono 16 kHz and convert it to fixed duration.
    Longer utterances are center-cropped.
    Shorter utterances are reflect-padded.

    If return_info=True, also return crop/padding metadata for reviewer analysis.
    """
    path = str(path)
    audio, _ = librosa.load(path, sr=sr, mono=True)

    original_len = int(len(audio))
    target_len = int(sr * duration)

    info = {
        "original_n_samples": original_len,
        "target_n_samples": target_len,
        "original_duration_sec": float(original_len / sr),
        "target_duration_sec": float(duration),
        "was_cropped": False,
        "was_padded": False,
        "crop_start_sample": 0,
        "crop_end_sample": original_len,
        "cropped_samples": 0,
        "padded_samples": 0,
        "cropped_seconds": 0.0,
        "padded_seconds": 0.0,
        "padding_mode": "none",
    }

    if original_len >= target_len:
        start = (original_len - target_len) // 2
        end = start + target_len
        audio = audio[start:end]

        info["was_cropped"] = bool(original_len > target_len)
        info["crop_start_sample"] = int(start)
        info["crop_end_sample"] = int(end)
        info["cropped_samples"] = int(max(0, original_len - target_len))
        info["cropped_seconds"] = float(max(0, original_len - target_len) / sr)
    else:
        pad_len = target_len - original_len
        if original_len > 1:
            audio = np.pad(audio, (0, pad_len), mode="reflect")
            padding_mode = "reflect"
        else:
            audio = np.pad(audio, (0, pad_len), mode="constant")
            padding_mode = "constant"

        info["was_padded"] = True
        info["padded_samples"] = int(pad_len)
        info["padded_seconds"] = float(pad_len / sr)
        info["padding_mode"] = padding_mode

    audio = audio.astype(np.float32)

    if return_info:
        return audio, info
    return audio


In [7]:
def parse_e2v_output(res):
    """
    Parse FunASR emotion2vec output robustly.

    We explicitly prioritize embedding-like keys. Classification scores are
    not accepted as embeddings unless no other option exists, and the final
    dimensionality check will stop the pipeline if a non-768 vector is returned.
    """
    if isinstance(res, list):
        item = res[0]
    else:
        item = res

    # Prefer embedding-like keys.
    possible_embedding_keys = [
        "feats",
        "feature",
        "embedding",
        "embeddings",
    ]

    if isinstance(item, dict):
        for key in possible_embedding_keys:
            if key in item:
                return np.asarray(item[key], dtype=np.float32)

        # Fallback: find the largest numeric array-like object, but avoid
        # obvious classification-score keys if possible.
        candidates = []
        for k, v in item.items():
            if str(k).lower() in ["scores", "score", "labels", "label"]:
                continue
            try:
                arr = np.asarray(v, dtype=np.float32)
                if arr.ndim >= 1 and arr.size > 10:
                    candidates.append((k, arr))
            except Exception:
                pass

        if len(candidates) > 0:
            candidates = sorted(candidates, key=lambda x: x[1].size, reverse=True)
            print(f"Fallback selected output key: {candidates[0][0]}")
            return candidates[0][1]

    raise ValueError(
        f"Cannot parse emotion2vec embedding output. "
        f"Available keys: {list(item.keys()) if isinstance(item, dict) else type(item)}"
    )


def extract_e2v_embedding_from_audio(audio, uid, tmp_dir, sr=16000):
    """Save temporary WAV and extract an utterance-level emotion2vec embedding."""
    tmp_path = tmp_dir / f"{uid}.wav"
    sf.write(tmp_path, audio, sr)

    res = e2v_model.generate(
        input=str(tmp_path),
        granularity=CONFIG["granularity"],
        extract_embedding=CONFIG["extract_embedding"],
    )

    emb = parse_e2v_output(res)
    emb = np.asarray(emb, dtype=np.float32)

    # Utterance output may be [D] or [1, D].
    # If frame-level output appears, mean-pool over time as a conservative fallback.
    if emb.ndim == 1:
        out = emb
    elif emb.ndim == 2:
        if emb.shape[0] == 1:
            out = emb[0]
        else:
            out = emb.mean(axis=0)
    elif emb.ndim == 3:
        out = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
    else:
        out = emb.reshape(-1)

    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    expected_dim = int(CONFIG["expected_embedding_dim"])
    if out.shape[0] != expected_dim:
        raise ValueError(
            f"Unexpected emotion2vec embedding dimension: got {out.shape[0]}, "
            f"expected {expected_dim}. This usually means the parser selected "
            f"classification scores instead of embeddings, or the model/config changed."
        )

    return out


In [8]:
# Smoke test on one EmoDB row before processing all files.
test_df = pd.read_csv(CSV_FILES["emodb"])
test_row = test_df.iloc[0]

audio, audio_info = load_audio_fixed(
    test_row["filepath"],
    sr=CONFIG["sample_rate"],
    duration=CONFIG["duration"],
    return_info=True,
)

emb = extract_e2v_embedding_from_audio(
    audio=audio,
    uid="test_e2v",
    tmp_dir=TMP_AUDIO_DIR,
    sr=CONFIG["sample_rate"],
)

print("Embedding shape:", emb.shape)
print("First 10 values:", emb[:10])
print("NaN:", np.isnan(emb).any())
print("Audio info:", audio_info)

if emb.shape[0] != CONFIG["expected_embedding_dim"]:
    raise RuntimeError("Smoke test failed: unexpected embedding dimension.")


rtf_avg: 0.364: 100%|██████████| 1/1 [00:01<00:00,  1.46s/it]

Embedding shape: (768,)
First 10 values: [-0.38277912  0.90200233 -0.15388812 -1.0022807  -0.18847048  0.7996264
 -0.43421695  1.0762302  -0.36453864  0.1717488 ]
NaN: False
Audio info: {'original_n_samples': 31304, 'target_n_samples': 64000, 'original_duration_sec': 1.9565, 'target_duration_sec': 4.0, 'was_cropped': False, 'was_padded': True, 'crop_start_sample': 0, 'crop_end_sample': 31304, 'cropped_samples': 0, 'padded_samples': 32696, 'cropped_seconds': 0.0, 'padded_seconds': 2.0435, 'padding_mode': 'reflect'}


In [9]:
def safe_uid(uid):
    return str(uid).replace("/", "_").replace("\\", "_").replace(" ", "_").replace(":", "_")


def extract_split_e2v(df_split, dataset_name, split_name):
    X = []
    rows = []
    errors = []

    split_tmp_dir = TMP_AUDIO_DIR / dataset_name / split_name
    split_tmp_dir.mkdir(parents=True, exist_ok=True)

    for idx, row in tqdm(
        df_split.iterrows(),
        total=len(df_split),
        desc=f"{dataset_name} {split_name} e2v"
    ):
        try:
            audio, audio_info = load_audio_fixed(
                row["filepath"],
                sr=CONFIG["sample_rate"],
                duration=CONFIG["duration"],
                return_info=True,
            )

            uid = safe_uid(row["uid"])

            emb = extract_e2v_embedding_from_audio(
                audio=audio,
                uid=uid,
                tmp_dir=split_tmp_dir,
                sr=CONFIG["sample_rate"],
            )

            X.append(emb)

            new_row = row.to_dict()
            new_row["is_augmented"] = False
            new_row.update(audio_info)
            rows.append(new_row)

        except Exception as e:
            errors.append({
                "idx": idx,
                "uid": row.get("uid", ""),
                "filepath": row.get("filepath", ""),
                "error": str(e),
                "split": split_name,
                "dataset": dataset_name,
            })

    if len(X) == 0:
        raise RuntimeError(f"No valid embeddings for {dataset_name} {split_name}")

    dims = [x.shape[0] for x in X]
    if len(set(dims)) != 1:
        raise ValueError(f"Inconsistent embedding dims in {dataset_name} {split_name}: {Counter(dims)}")

    X = np.asarray(X, dtype=np.float32)
    meta = pd.DataFrame(rows)
    err = pd.DataFrame(errors)

    y = meta["emotion"].map(LABEL_TO_ID).values.astype(np.int64)

    return X, y, meta, err


In [10]:
def summarize_duration_crop_padding(meta_df, group_cols):
    numeric_cols = [
        "original_duration_sec",
        "was_cropped",
        "was_padded",
        "cropped_seconds",
        "padded_seconds",
    ]

    df = meta_df.copy()
    df["was_cropped"] = df["was_cropped"].astype(int)
    df["was_padded"] = df["was_padded"].astype(int)

    summary = (
        df.groupby(group_cols)
        .agg(
            n=("uid", "count"),
            duration_mean=("original_duration_sec", "mean"),
            duration_std=("original_duration_sec", "std"),
            duration_median=("original_duration_sec", "median"),
            duration_min=("original_duration_sec", "min"),
            duration_max=("original_duration_sec", "max"),
            cropped_n=("was_cropped", "sum"),
            padded_n=("was_padded", "sum"),
            cropped_pct=("was_cropped", lambda x: 100.0 * float(np.mean(x))),
            padded_pct=("was_padded", lambda x: 100.0 * float(np.mean(x))),
            cropped_seconds_mean=("cropped_seconds", "mean"),
            padded_seconds_mean=("padded_seconds", "mean"),
        )
        .reset_index()
    )
    return summary


def process_one_dataset_e2v(dataset_name, csv_path):
    print("=" * 90)
    print(f"Processing emotion2vec: {dataset_name.upper()}")
    print("=" * 90)

    out_ds = OUT_DIR / dataset_name
    out_ds.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(csv_path)
    df = df[df["emotion"].isin(LABELS)].copy()
    df["label"] = df["emotion"].map(LABEL_TO_ID).astype(int)

    print("Rows:", len(df))
    print("Speakers:", df["speaker"].nunique())
    print("\nDistribution:")
    display(df.groupby(["split", "emotion"]).size().unstack(fill_value=0))

    outputs = {}
    all_errors = []

    for split_name in ["train", "val", "test"]:
        df_split = df[df["split"] == split_name].reset_index(drop=True)

        X, y, meta, err = extract_split_e2v(
            df_split=df_split,
            dataset_name=dataset_name,
            split_name=split_name
        )

        outputs[split_name] = {
            "X": X,
            "y": y,
            "meta": meta,
            "err": err,
        }

        print(f"{split_name}: X={X.shape}, y={y.shape}, errors={len(err)}")

        np.save(out_ds / f"X_e2v_{split_name}.npy", X)
        np.save(out_ds / f"y_{split_name}.npy", y)
        meta.to_csv(out_ds / f"meta_{split_name}.csv", index=False)

        if len(err) > 0:
            all_errors.append(err)

    if len(all_errors) > 0:
        errors_df = pd.concat(all_errors, ignore_index=True)
    else:
        errors_df = pd.DataFrame(columns=["idx", "uid", "filepath", "error", "split", "dataset"])

    errors_df.to_csv(out_ds / "errors.csv", index=False)

    all_meta = pd.concat(
        [outputs[split_name]["meta"].assign(split=split_name) for split_name in ["train", "val", "test"]],
        ignore_index=True,
    )

    duration_stats_split_emotion = summarize_duration_crop_padding(
        all_meta,
        group_cols=["split", "emotion"],
    )
    duration_stats_emotion = summarize_duration_crop_padding(
        all_meta,
        group_cols=["emotion"],
    )
    duration_stats_overall = summarize_duration_crop_padding(
        all_meta.assign(all="all"),
        group_cols=["all"],
    )

    duration_stats_split_emotion.to_csv(out_ds / "duration_crop_padding_stats_by_split_emotion.csv", index=False)
    duration_stats_emotion.to_csv(out_ds / "duration_crop_padding_stats_by_emotion.csv", index=False)
    duration_stats_overall.to_csv(out_ds / "duration_crop_padding_stats_overall.csv", index=False)

    feature_config = {
        **CONFIG,
        "dataset": dataset_name,
        "embedding_dim": int(outputs["train"]["X"].shape[1]),
        "n_train": int(outputs["train"]["X"].shape[0]),
        "n_val": int(outputs["val"]["X"].shape[0]),
        "n_test": int(outputs["test"]["X"].shape[0]),
        "label_to_id": LABEL_TO_ID,
        "note": (
            "emotion2vec-plus-base utterance-level embeddings; no augmentation; "
            "no scaler applied at extraction stage; audio fixed to 4 seconds by "
            "center cropping or reflect padding."
        ),
    }

    with open(out_ds / "feature_config.json", "w") as f:
        json.dump(feature_config, f, indent=2)

    print("\nSaved:", out_ds)
    print("Embedding dim:", outputs["train"]["X"].shape[1])
    print("Errors:", len(errors_df))

    return {
        "dataset": dataset_name,
        "embedding_dim": outputs["train"]["X"].shape[1],
        "n_train": outputs["train"]["X"].shape[0],
        "n_val": outputs["val"]["X"].shape[0],
        "n_test": outputs["test"]["X"].shape[0],
        "n_errors": len(errors_df),
        "duration_mean": float(all_meta["original_duration_sec"].mean()),
        "cropped_pct": float(100.0 * all_meta["was_cropped"].mean()),
        "padded_pct": float(100.0 * all_meta["was_padded"].mean()),
    }


In [11]:
summary_rows = []

for dataset_name in DATASETS:
    result = process_one_dataset_e2v(
        dataset_name=dataset_name,
        csv_path=CSV_FILES[dataset_name]
    )
    summary_rows.append(result)

summary_e2v = pd.DataFrame(summary_rows)
summary_e2v.to_csv(OUT_DIR / "emotion2vec_extraction_summary.csv", index=False)

display(summary_e2v)
print("Saved:", OUT_DIR / "emotion2vec_extraction_summary.csv")


Processing emotion2vec: EMODB
Rows: 718
Speakers: 10

Distribution:


emotion,angry,disgust,fear,happy,neutral,sad
split,,,,,,
test,29,22,25,24,22,27
train,98,73,85,82,74,86
val,13,11,13,12,10,12


  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.014', 'extract_feat': 0.0, 'forward': '0.029', 'batch_size': '1', 'rtf': '0.007'}, : 100%|██████████| 1/1 [00:00<00:00, 33.84it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.013', 'extract_feat': 0.0, 'forward': '0.027', 'batch_size': '1', 'rtf': '0.007'}, : 100%|██████████| 1/1 [00:00<00:00, 35.42it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.014', 'extract_feat': 0.0, 'forward': '0.028', 'batch_size': '1', 'rtf': '0.007'}, : 100%|██████████| 1/1 [00:00<00:00, 34.93it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.013', 'extract_feat': 0.0, 'forward': '0.027', 'batch_size': '1', 'rtf': '0.007'}, : 100%|██████████| 1/1 [00:00<00:00, 35.51it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.013', 'extract_feat': 0.0, 'forward': '0.027', 'batch_size': '1', 'rtf': '0.007'}, : 100%|██████████| 1/1 [00:00<00:00, 36.23it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.013', 'extract_

train: X=(498, 768), y=(498,), errors=0


  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.013', 'extract_feat': 0.0, 'forward': '0.028', 'batch_size': '1', 'rtf': '0.007'}, : 100%|██████████| 1/1 [00:00<00:00, 34.84it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.014', 'extract_feat': 0.0, 'forward': '0.029', 'batch_size': '1', 'rtf': '0.007'}, : 100%|██████████| 1/1 [00:00<00:00, 33.29it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.013', 'extract_feat': 0.0, 'forward': '0.028', 'batch_size': '1', 'rtf': '0.007'}, : 100%|██████████| 1/1 [00:00<00:00, 35.43it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.013', 'extract_feat': 0.0, 'forward': '0.028', 'batch_size': '1', 'rtf': '0.007'}, : 100%|██████████| 1/1 [00:00<00:00, 35.14it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.013', 'extract_feat': 0.0, 'forward': '0.031', 'batch_size': '1', 'rtf': '0.008'}, : 100%|██████████| 1/1 [00:00<00:00, 31.19it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.013', 'extract_

...

val: X=(186, 768), y=(186,), errors=0


  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.015', 'extract_feat': 0.0, 'forward': '0.030', 'batch_size': '1', 'rtf': '0.008'}, : 100%|██████████| 1/1 [00:00<00:00, 32.36it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.013', 'extract_feat': 0.0, 'forward': '0.032', 'batch_size': '1', 'rtf': '0.008'}, : 100%|██████████| 1/1 [00:00<00:00, 30.29it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.016', 'extract_feat': 0.0, 'forward': '0.031', 'batch_size': '1', 'rtf': '0.008'}, : 100%|██████████| 1/1 [00:00<00:00, 31.97it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.013', 'extract_feat': 0.0, 'forward': '0.028', 'batch_size': '1', 'rtf': '0.007'}, : 100%|██████████| 1/1 [00:00<00:00, 34.62it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.018', 'extract_feat': 0.0, 'forward': '0.035', 'batch_size': '1', 'rtf': '0.009'}, : 100%|██████████| 1/1 [00:00<00:00, 28.17it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.014', 'extract_

test: X=(139, 768), y=(139,), errors=0

Saved: /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_e2v_plus_base/resd
Embedding dim: 768
Errors: 0


,dataset,embedding_dim,n_train,n_val,n_test,n_errors,duration_mean,cropped_pct,padded_pct
0,emodb,768,498,71,149,0,2.768886,9.749304,90.250696
1,ravdess,768,704,176,176,0,3.722143,18.750000,81.250000
2,resd,768,873,186,139,0,6.098272,59.599332,40.400668


Saved: /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_e2v_plus_base/emotion2vec_extraction_summary.csv


In [12]:
# Combined duration/crop/padding statistics across all datasets.
combined_meta = []

for dataset_name in DATASETS:
    ds_dir = OUT_DIR / dataset_name
    for split in ["train", "val", "test"]:
        meta = pd.read_csv(ds_dir / f"meta_{split}.csv")
        meta["dataset"] = dataset_name
        meta["split"] = split
        combined_meta.append(meta)

combined_meta = pd.concat(combined_meta, ignore_index=True)

duration_stats_dataset_emotion = summarize_duration_crop_padding(
    combined_meta,
    group_cols=["dataset", "emotion"],
)
duration_stats_dataset_split = summarize_duration_crop_padding(
    combined_meta,
    group_cols=["dataset", "split"],
)
duration_stats_dataset = summarize_duration_crop_padding(
    combined_meta,
    group_cols=["dataset"],
)

duration_stats_dataset_emotion.to_csv(OUT_DIR / "duration_crop_padding_stats_by_dataset_emotion.csv", index=False)
duration_stats_dataset_split.to_csv(OUT_DIR / "duration_crop_padding_stats_by_dataset_split.csv", index=False)
duration_stats_dataset.to_csv(OUT_DIR / "duration_crop_padding_stats_by_dataset.csv", index=False)

display(duration_stats_dataset)
print("Saved duration/crop/padding summary files in:", OUT_DIR)


,dataset,n,duration_mean,duration_std,duration_median,duration_min,duration_max,cropped_n,padded_n,cropped_pct,padded_pct,cropped_seconds_mean,padded_seconds_mean
0,emodb,718,2.768886,1.042156,2.568781,1.213375,8.978250,70,648,9.749304,90.250696,0.101560,1.332674
1,ravdess,1056,3.722143,0.337217,3.670375,3.069750,5.271937,198,858,18.750000,81.250000,0.048339,0.326197
2,resd,1198,6.098272,3.964249,5.148781,0.370000,20.139312,714,484,59.599332,40.400668,2.713720,0.615447


Saved duration/crop/padding summary files in: /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_e2v_plus_base


In [13]:
# Save SHA256 checksums for generated feature/config files.
feature_manifest_rows = []
suffixes = {".npy", ".csv", ".json"}

for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file() and path.suffix.lower() in suffixes:
        # Avoid hashing this manifest while it is being generated.
        if path.name == "generated_feature_file_manifest_sha256.csv":
            continue
        feature_manifest_rows.append({
            "relative_path": str(path.relative_to(OUT_DIR)),
            "absolute_path": str(path),
            "size_bytes": int(path.stat().st_size),
            "sha256": sha256_of_file(path),
        })

feature_file_manifest = pd.DataFrame(feature_manifest_rows)
feature_file_manifest.to_csv(OUT_DIR / "generated_feature_file_manifest_sha256.csv", index=False)

display(feature_file_manifest.head(20))
print("Saved:", OUT_DIR / "generated_feature_file_manifest_sha256.csv")
print("Number of files hashed:", len(feature_file_manifest))


,relative_path,absolute_path,size_bytes,sha256
0,duration_crop_padding_stats_by_dataset.csv,/content/drive/MyDrive/New Jurnal Cross/proces...,608,59847355c88959a0d018334e4a72d2667846a91a26bff1...
1,duration_crop_padding_stats_by_dataset_emotion...,/content/drive/MyDrive/New Jurnal Cross/proces...,2939,1d7490e4a42f3e6eb3ce25ff55bcb648c5594991457b28...
2,duration_crop_padding_stats_by_dataset_split.csv,/content/drive/MyDrive/New Jurnal Cross/proces...,1629,2b9418916ff886e22973c5daf6bb81ab72b08eeeedfc5b...
3,emodb/X_e2v_test.npy,/content/drive/MyDrive/New Jurnal Cross/proces...,457856,7148f2b153de9715d62414906c5b8500a4099be7f50f3b...
4,emodb/X_e2v_train.npy,/content/drive/MyDrive/New Jurnal Cross/proces...,1529984,4eeae3e88bc83f5b7bd3f3df87da192a2a5fd63fca757c...
5,emodb/X_e2v_val.npy,/content/drive/MyDrive/New Jurnal Cross/proces...,218240,48d88eb77e6bbb12deec43337d059eeaa3933f4dda3127...
6,emodb/duration_crop_padding_stats_by_emotion.csv,/content/drive/MyDrive/New Jurnal Cross/proces...,1007,aa8fe89a68029bd54ecb3d891ac7c8f4128058056729d5...
7,emodb/duration_crop_padding_stats_by_split_emo...,/content/drive/MyDrive/New Jurnal Cross/proces...,2719,134c44c338d5a64ce7ae1ed0e87742f4e9313a975eaf3c...
8,emodb/duration_crop_padding_stats_overall.csv,/content/drive/MyDrive/New Jurnal Cross/proces...,313,c56a87a343964f91d3ce80b081e1f0dde441556a54f5d7...
9,emodb/errors.csv,/content/drive/MyDrive/New Jurnal Cross/proces...,37,f98540dfb13c3a245613a5a867b1461600293818650b2a...


Saved: /content/drive/MyDrive/New Jurnal Cross/processed_intra_features_e2v_plus_base/generated_feature_file_manifest_sha256.csv
Number of files hashed: 47


In [14]:
# Final verification.
for dataset_name in DATASETS:
    ds_dir = OUT_DIR / dataset_name

    print("=" * 80)
    print(dataset_name.upper())

    for split in ["train", "val", "test"]:
        X = np.load(ds_dir / f"X_e2v_{split}.npy")
        y = np.load(ds_dir / f"y_{split}.npy")
        meta = pd.read_csv(ds_dir / f"meta_{split}.csv")

        print(f"{split:5s} X={X.shape} y={y.shape} meta={meta.shape} NaN={np.isnan(X).any()}")

        if X.shape[1] != CONFIG["expected_embedding_dim"]:
            raise RuntimeError(
                f"{dataset_name} {split}: expected dim {CONFIG['expected_embedding_dim']}, got {X.shape[1]}"
            )

        print(pd.Series(y).value_counts().sort_index().rename(index=ID_TO_LABEL).to_dict())

print("\nAll checks passed.")


EMODB
train X=(498, 768) y=(498,) meta=(498, 42) NaN=False
{'angry': 98, 'disgust': 73, 'fear': 85, 'happy': 82, 'neutral': 74, 'sad': 86}
val   X=(71, 768) y=(71,) meta=(71, 42) NaN=False
{'angry': 13, 'disgust': 11, 'fear': 13, 'happy': 12, 'neutral': 10, 'sad': 12}
test  X=(149, 768) y=(149,) meta=(149, 42) NaN=False
{'angry': 29, 'disgust': 22, 'fear': 25, 'happy': 24, 'neutral': 22, 'sad': 27}
RAVDESS
train X=(704, 768) y=(704,) meta=(704, 42) NaN=False
{'angry': 128, 'disgust': 128, 'fear': 128, 'happy': 128, 'neutral': 64, 'sad': 128}
val   X=(176, 768) y=(176,) meta=(176, 42) NaN=False
{'angry': 32, 'disgust': 32, 'fear': 32, 'happy': 32, 'neutral': 16, 'sad': 32}
test  X=(176, 768) y=(176,) meta=(176, 42) NaN=False
{'angry': 32, 'disgust': 32, 'fear': 32, 'happy': 32, 'neutral': 16, 'sad': 32}
RESD
train X=(873, 768) y=(873,) meta=(873, 42) NaN=False
{'angry': 154, 'disgust': 135, 'fear': 162, 'happy': 160, 'neutral': 140, 'sad': 122}
val   X=(186, 768) y=(186,) meta=(186, 42)